# Zone 2 (Below Center) Bucket Analysis

Implements 7 modeling approaches to predict Zone 2 bucket probabilities.
Zone 2 = relative_position 20-40% (below center). Market overprices by ~0.4%.

In [ ]:
import json
import math
import warnings
from pathlib import Path

import numpy as np
from scipy import optimize, stats

warnings.filterwarnings('ignore')

PROJECT = Path(r'G:\My Drive\AI_Projects\Ideas - In Progress\ElonTweetMarkets')
DATA_PATH = PROJECT / 'data' / 'analysis' / 'bucket_dataset.json'
OUTPUT_PATH = PROJECT / 'data' / 'analysis' / 'zone_2_results.json'

TARGET_ZONE = 2
PROB_FLOOR = 1e-6
print('Setup complete')

In [ ]:
def normalize_probs(probs):
    floored = [max(p, PROB_FLOOR) for p in probs]
    total = sum(floored)
    if total <= 0:
        n = len(floored)
        return [1.0 / n] * n
    return [p / total for p in floored]

def brier_score(probs, outcomes):
    return sum((p - o) ** 2 for p, o in zip(probs, outcomes)) / len(probs)

def zone_brier(probs, outcomes, zones, target_zone):
    pairs = [(p, o) for p, o, z in zip(probs, outcomes, zones) if z == target_zone]
    if not pairs:
        return float('nan')
    return sum((p - o) ** 2 for p, o in pairs) / len(pairs)

def sigmoid(x):
    if x > 500: return 1.0
    if x < -500: return 0.0
    return 1.0 / (1.0 + math.exp(-x))

def logit(p):
    p = max(min(p, 1.0 - 1e-10), 1e-10)
    return math.log(p / (1.0 - p))

def safe_float(val):
    if val is None: return None
    try:
        f = float(val)
        if math.isnan(f) or math.isinf(f): return None
        return f
    except (TypeError, ValueError): return None

print('Helpers defined')

In [ ]:
with open(DATA_PATH, encoding='utf-8') as f:
    dataset = json.load(f)

events = []
for evt in dataset['events']:
    buckets = evt['buckets']
    if not buckets:
        continue
    zones = [b['zone'] for b in buckets]
    if TARGET_ZONE not in zones:
        continue
    features = evt.get('features', {})
    market_feats = features.get('market', {})
    temporal_feats = features.get('temporal', {})
    media_feats = features.get('media', {})
    calendar_feats = features.get('calendar', {})
    events.append({
        'slug': evt['event_slug'],
        'market_type': evt.get('market_type'),
        'duration_days': evt.get('duration_days'),
        'tier': evt.get('ground_truth_tier'),
        'xtracker_count': evt.get('xtracker_count'),
        'n_buckets': len(buckets),
        'market_prices': [b['market_price'] for b in buckets],
        'outcomes': [b['is_winner'] for b in buckets],
        'zones': zones,
        'z_scores': [b.get('z_score') for b in buckets],
        'lower_bounds': [b['lower_bound'] for b in buckets],
        'upper_bounds': [b['upper_bound'] for b in buckets],
        'midpoints': [b['midpoint'] for b in buckets],
        'buckets_raw': buckets,
        'crowd_ev': safe_float(market_feats.get('crowd_implied_ev')),
        'crowd_std': safe_float(market_feats.get('crowd_std_dev')),
        'crowd_skewness': safe_float(market_feats.get('crowd_skewness')),
        'crowd_kurtosis': safe_float(market_feats.get('crowd_kurtosis')),
        'distribution_entropy': safe_float(market_feats.get('distribution_entropy')),
        'price_shift_24h': safe_float(market_feats.get('price_shift_24h')),
        'rolling_avg_7d': safe_float(temporal_feats.get('rolling_avg_7d')),
        'trend_7d': safe_float(temporal_feats.get('trend_7d')),
        'rolling_std_7d': safe_float(temporal_feats.get('rolling_std_7d')),
        'elon_musk_vol_7d': safe_float(media_feats.get('elon_musk_vol_7d')),
        'elon_musk_tone_7d': safe_float(media_feats.get('elon_musk_tone_7d')),
        'launches_trailing_7d': safe_float(calendar_feats.get('launches_trailing_7d')),
    })

n_z2 = sum(1 for e in events for j, z in enumerate(e['zones']) if z == TARGET_ZONE)
n_z2_w = sum(1 for e in events for j, z in enumerate(e['zones']) if z == TARGET_ZONE and e['outcomes'][j])
tier_counts = {}
for e in events:
    t = e['tier'] or 'unknown'
    tier_counts[t] = tier_counts.get(t, 0) + 1

print(f'Events: {len(events)}, Zone 2 buckets: {n_z2}, winners: {n_z2_w} ({n_z2_w/n_z2:.3f})')
print(f'Tiers: {tier_counts}')

In [ ]:
# Approach 1: NegBin from crowd stats
full_b1, zone_b1 = [], []
for evt in events:
    ce, cs = evt['crowd_ev'], evt['crowd_std']
    if ce and cs and cs > 0 and ce > 0:
        var = cs ** 2
        if var <= ce:
            probs_raw = [max(stats.poisson.cdf(min(ub, int(ce*10)), ce) - stats.poisson.cdf(max(lb-1,0), ce), PROB_FLOOR)
                         for lb, ub in zip(evt['lower_bounds'], evt['upper_bounds'])]
        else:
            p_nb = ce / var
            n_nb = ce**2 / (var - ce)
            if n_nb > 0 and 0 < p_nb < 1:
                probs_raw = [max(stats.nbinom.cdf(min(ub,100000), n_nb, p_nb) - stats.nbinom.cdf(max(lb-1,0), n_nb, p_nb), PROB_FLOOR)
                             for lb, ub in zip(evt['lower_bounds'], evt['upper_bounds'])]
            else:
                probs_raw = evt['market_prices']
        probs = normalize_probs(probs_raw)
    else:
        probs = normalize_probs(evt['market_prices'])
    full_b1.append(brier_score(probs, evt['outcomes']))
    zone_b1.append(zone_brier(probs, evt['outcomes'], evt['zones'], TARGET_ZONE))

r1 = {'name': 'NegBin from crowd stats', 'full_brier': float(np.mean(full_b1)),
       'zone_brier': float(np.nanmean(zone_b1)), 'learned': False}
print(f"Approach 1 - Full: {r1['full_brier']:.6f}, Zone: {r1['zone_brier']:.6f}")

In [ ]:
# Approach 2: Gaussian from crowd stats
full_b2, zone_b2 = [], []
for evt in events:
    ce, cs = evt['crowd_ev'], evt['crowd_std']
    if ce and cs and cs > 0:
        dist = stats.norm(loc=ce, scale=cs)
        probs_raw = []
        for lb, ub in zip(evt['lower_bounds'], evt['upper_bounds']):
            if ub >= 99999:
                p = 1.0 - dist.cdf(lb - 0.5)
            else:
                p = dist.cdf(ub + 0.5) - dist.cdf(lb - 0.5)
            probs_raw.append(max(p, PROB_FLOOR))
        probs = normalize_probs(probs_raw)
    else:
        probs = normalize_probs(evt['market_prices'])
    full_b2.append(brier_score(probs, evt['outcomes']))
    zone_b2.append(zone_brier(probs, evt['outcomes'], evt['zones'], TARGET_ZONE))

r2 = {'name': 'Gaussian from crowd stats', 'full_brier': float(np.mean(full_b2)),
       'zone_brier': float(np.nanmean(zone_b2)), 'learned': False}
print(f"Approach 2 - Full: {r2['full_brier']:.6f}, Zone: {r2['zone_brier']:.6f}")

In [ ]:
# Approach 3: Market baseline
full_b3, zone_b3 = [], []
for evt in events:
    probs = normalize_probs(evt['market_prices'])
    full_b3.append(brier_score(probs, evt['outcomes']))
    zone_b3.append(zone_brier(probs, evt['outcomes'], evt['zones'], TARGET_ZONE))

r3 = {'name': 'Market baseline', 'full_brier': float(np.mean(full_b3)),
       'zone_brier': float(np.nanmean(zone_b3)), 'learned': False}
print(f"Approach 3 - Full: {r3['full_brier']:.6f}, Zone: {r3['zone_brier']:.6f}")

In [ ]:
# Approach 4: Platt-calibrated market (LOOCV)
full_b4, zone_b4, base_b4 = [], [], []

for hold_idx in range(len(events)):
    train_logits, train_labels = [], []
    for i, evt in enumerate(events):
        if i == hold_idx: continue
        for j, z in enumerate(evt['zones']):
            if z == TARGET_ZONE:
                mp = evt['market_prices'][j]
                if mp > PROB_FLOOR:
                    train_logits.append(logit(mp))
                    train_labels.append(evt['outcomes'][j])

    if len(train_logits) < 5 or sum(train_labels) < 1:
        evt_test = events[hold_idx]
        probs = normalize_probs(evt_test['market_prices'])
        full_b4.append(brier_score(probs, evt_test['outcomes']))
        zone_b4.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
        base_b4.append(brier_score(probs, evt_test['outcomes']))
        continue

    X = np.array(train_logits)
    y = np.array(train_labels, dtype=float)

    def neg_ll(params):
        a, b = params
        preds = 1.0 / (1.0 + np.exp(-(a * X + b)))
        preds = np.clip(preds, 1e-10, 1 - 1e-10)
        return -np.sum(y * np.log(preds) + (1-y) * np.log(1-preds))

    try:
        res = optimize.minimize(neg_ll, [1.0, 0.0], method='Nelder-Mead', options={'maxiter': 2000})
        a_opt, b_opt = res.x
    except:
        a_opt, b_opt = 1.0, 0.0

    evt_test = events[hold_idx]
    adj = list(evt_test['market_prices'])
    for j, z in enumerate(evt_test['zones']):
        if z == TARGET_ZONE:
            mp = evt_test['market_prices'][j]
            adj[j] = sigmoid(a_opt * logit(mp) + b_opt) if mp > PROB_FLOOR else PROB_FLOOR

    probs = normalize_probs(adj)
    full_b4.append(brier_score(probs, evt_test['outcomes']))
    zone_b4.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
    base_b4.append(brier_score(normalize_probs(evt_test['market_prices']), evt_test['outcomes']))

r4 = {'name': 'Platt-calibrated market', 'full_brier': float(np.mean(full_b4)),
       'zone_brier': float(np.nanmean(zone_b4)),
       'events_beating_baseline': sum(1 for f, b in zip(full_b4, base_b4) if f < b), 'learned': True}
print(f"Approach 4 - Full: {r4['full_brier']:.6f}, Zone: {r4['zone_brier']:.6f}, Beats: {r4['events_beating_baseline']}")

In [ ]:
# Approach 5: Market + downswing correction (LOOCV)
full_b5, zone_b5, base_b5, shrinks = [], [], [], []

for hold_idx in range(len(events)):
    train = [e for i, e in enumerate(events) if i != hold_idx]

    def eval_shrink(shrink):
        tb = 0
        for evt in train:
            adj = list(evt['market_prices'])
            rm, nzm = 0, 0
            for j, z in enumerate(evt['zones']):
                if z == TARGET_ZONE:
                    red = adj[j] * shrink
                    adj[j] -= red
                    rm += red
                else:
                    nzm += adj[j]
            if nzm > 0 and rm > 0:
                for j, z in enumerate(evt['zones']):
                    if z != TARGET_ZONE:
                        adj[j] += rm * (adj[j] / nzm)
            tb += brier_score(normalize_probs(adj), evt['outcomes'])
        return tb / len(train)

    best_s, best_tb = 0, eval_shrink(0)
    for s in np.arange(0.0, 0.50, 0.01):
        tb = eval_shrink(s)
        if tb < best_tb:
            best_tb = tb
            best_s = s
    shrinks.append(best_s)

    evt_test = events[hold_idx]
    adj = list(evt_test['market_prices'])
    rm, nzm = 0, 0
    for j, z in enumerate(evt_test['zones']):
        if z == TARGET_ZONE:
            red = adj[j] * best_s
            adj[j] -= red
            rm += red
        else:
            nzm += adj[j]
    if nzm > 0 and rm > 0:
        for j, z in enumerate(evt_test['zones']):
            if z != TARGET_ZONE:
                adj[j] += rm * (adj[j] / nzm)

    probs = normalize_probs(adj)
    full_b5.append(brier_score(probs, evt_test['outcomes']))
    zone_b5.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
    base_b5.append(brier_score(normalize_probs(evt_test['market_prices']), evt_test['outcomes']))

r5 = {'name': 'Market + downswing correction', 'full_brier': float(np.mean(full_b5)),
       'zone_brier': float(np.nanmean(zone_b5)),
       'events_beating_baseline': sum(1 for f, b in zip(full_b5, base_b5) if f < b),
       'avg_shrink_factor': float(np.mean(shrinks)), 'learned': True}
print(f"Approach 5 - Full: {r5['full_brier']:.6f}, Zone: {r5['zone_brier']:.6f}, Beats: {r5['events_beating_baseline']}, Avg shrink: {r5['avg_shrink_factor']:.4f}")

In [ ]:
# Approach 6: Market + crowd-EV-vs-rolling-avg correction (LOOCV)

def compute_adj_factor(evt, alpha_t, alpha_m):
    factor = 0.0
    duration = evt.get('duration_days') or 7
    ra = evt.get('rolling_avg_7d')
    ce = evt.get('crowd_ev')
    if ra is not None and ce is not None and ce > 0:
        expected = ra * duration
        gap = (expected - ce) / ce
        gap = max(min(gap, 1.0), -1.0)
        factor += alpha_t * gap
    else:
        vol = evt.get('elon_musk_vol_7d')
        if vol is not None:
            factor += alpha_m * (-(vol - 0.3))
    return factor

def apply_zone_adj(evt, adj_factor):
    adj = list(evt['market_prices'])
    rm, nzm = 0, 0
    for j, z in enumerate(evt['zones']):
        if z == TARGET_ZONE:
            old_p = adj[j]
            new_p = max(old_p * (1.0 + adj_factor), PROB_FLOOR)
            rm += old_p - new_p
            adj[j] = new_p
        else:
            nzm += adj[j]
    if nzm > 0 and abs(rm) > PROB_FLOOR:
        for j, z in enumerate(evt['zones']):
            if z != TARGET_ZONE:
                adj[j] = max(adj[j] + rm * (adj[j] / nzm), PROB_FLOOR)
    return adj

full_b6, zone_b6, base_b6 = [], [], []
for hold_idx in range(len(events)):
    train = [e for i, e in enumerate(events) if i != hold_idx]

    def eval_params(at, am):
        tb = 0
        for evt in train:
            af = compute_adj_factor(evt, at, am)
            adj = apply_zone_adj(evt, af)
            tb += brier_score(normalize_probs(adj), evt['outcomes'])
        return tb / len(train)

    best_at, best_am, best_tb = 0, 0, eval_params(0, 0)
    for at in np.arange(-0.3, 0.31, 0.05):
        for am in np.arange(-0.3, 0.31, 0.05):
            tb = eval_params(at, am)
            if tb < best_tb:
                best_tb = tb
                best_at, best_am = at, am

    evt_test = events[hold_idx]
    af = compute_adj_factor(evt_test, best_at, best_am)
    adj = apply_zone_adj(evt_test, af)
    probs = normalize_probs(adj)
    full_b6.append(brier_score(probs, evt_test['outcomes']))
    zone_b6.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
    base_b6.append(brier_score(normalize_probs(evt_test['market_prices']), evt_test['outcomes']))

r6 = {'name': 'Market + crowd-EV-vs-rolling correction', 'full_brier': float(np.mean(full_b6)),
       'zone_brier': float(np.nanmean(zone_b6)),
       'events_beating_baseline': sum(1 for f, b in zip(full_b6, base_b6) if f < b), 'learned': True}
print(f"Approach 6 - Full: {r6['full_brier']:.6f}, Zone: {r6['zone_brier']:.6f}, Beats: {r6['events_beating_baseline']}")

In [ ]:
# Approach 7: Multi-feature adjustment (LOOCV)

def build_fv(evt, j):
    b = evt['buckets_raw'][j]
    zs = safe_float(b.get('z_score'))
    return [
        zs if zs is not None else 0.0,
        b.get('market_price', 0),
        evt.get('crowd_skewness') or 0.0,
        evt.get('crowd_kurtosis') or 0.0,
        evt.get('price_shift_24h') or 0.0,
        evt.get('elon_musk_tone_7d') or 0.0,
        evt.get('launches_trailing_7d') or 0.0,
        evt.get('distribution_entropy') or 0.0,
    ]

# Pre-compute features
evt_feats = []
for evt in events:
    ef = [(j, build_fv(evt, j)) for j, z in enumerate(evt['zones']) if z == TARGET_ZONE]
    evt_feats.append(ef)

full_b7, zone_b7, base_b7 = [], [], []
for hold_idx in range(len(events)):
    train_X, train_y = [], []
    for i, evt in enumerate(events):
        if i == hold_idx: continue
        for j, fv in evt_feats[i]:
            if fv is not None:
                train_X.append(fv)
                train_y.append(evt['outcomes'][j] - evt['market_prices'][j])

    if len(train_X) < 10:
        evt_test = events[hold_idx]
        probs = normalize_probs(evt_test['market_prices'])
        full_b7.append(brier_score(probs, evt_test['outcomes']))
        zone_b7.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
        base_b7.append(brier_score(probs, evt_test['outcomes']))
        continue

    X = np.array(train_X)
    y = np.array(train_y)
    Xm, Xs = X.mean(0), X.std(0)
    Xs[Xs < 1e-10] = 1.0
    Xn = (X - Xm) / Xs
    lam = 1.0
    nf = Xn.shape[1]
    try:
        w = np.linalg.solve(Xn.T @ Xn + lam * np.eye(nf), Xn.T @ y)
    except:
        w = np.zeros(nf)

    evt_test = events[hold_idx]
    adj = list(evt_test['market_prices'])
    rm, nzm = 0, 0
    for j, fv in evt_feats[hold_idx]:
        if fv is not None:
            fn = (np.array(fv) - Xm) / Xs
            rp = float(fn @ w)
            rp = max(min(rp, 0.15), -0.15)
            old_p = adj[j]
            adj[j] = max(old_p + rp, PROB_FLOOR)
            rm += old_p - adj[j]

    nzm = sum(adj[j] for j in range(len(adj)) if evt_test['zones'][j] != TARGET_ZONE)
    if nzm > 0 and abs(rm) > PROB_FLOOR:
        for j, z in enumerate(evt_test['zones']):
            if z != TARGET_ZONE:
                adj[j] = max(adj[j] + rm * (adj[j] / nzm), PROB_FLOOR)

    probs = normalize_probs(adj)
    full_b7.append(brier_score(probs, evt_test['outcomes']))
    zone_b7.append(zone_brier(probs, evt_test['outcomes'], evt_test['zones'], TARGET_ZONE))
    base_b7.append(brier_score(normalize_probs(evt_test['market_prices']), evt_test['outcomes']))

r7 = {'name': 'Multi-feature adjustment', 'full_brier': float(np.mean(full_b7)),
       'zone_brier': float(np.nanmean(zone_b7)),
       'events_beating_baseline': sum(1 for f, b in zip(full_b7, base_b7) if f < b), 'learned': True}
print(f"Approach 7 - Full: {r7['full_brier']:.6f}, Zone: {r7['zone_brier']:.6f}, Beats: {r7['events_beating_baseline']}")

In [ ]:
# Summary and save results
results = [r1, r2, r3, r4, r5, r6, r7]
baseline_brier = r3['full_brier']
best = min(results, key=lambda r: r['full_brier'])

print('=' * 75)
print('ZONE 2 ANALYSIS SUMMARY')
print('=' * 75)
print(f"{'Approach':<42} {'Full Brier':>11} {'Zone Brier':>11} {'vs Base':>10}")
print('-' * 75)
for r in sorted(results, key=lambda x: x['full_brier']):
    delta = r['full_brier'] - baseline_brier
    zb = f"{r['zone_brier']:.6f}" if r.get('zone_brier') is not None else 'N/A'
    marker = ' ***' if r['name'] == best['name'] else ''
    print(f"  {r['name']:<40} {r['full_brier']:.6f}   {zb}  {delta:+.6f}{marker}")

print(f"\nBaseline: {baseline_brier:.6f}")
print(f"Best: {best['name']} ({best['full_brier']:.6f})")
print(f"Improvement: {baseline_brier - best['full_brier']:.6f}")

# Save
output = {
    'zone': TARGET_ZONE,
    'n_events': len(events),
    'n_zone_buckets': n_z2,
    'n_zone_winners': n_z2_w,
    'zone_win_rate': round(n_z2_w / n_z2, 4) if n_z2 > 0 else 0,
    'approaches': results,
    'best_approach': best['name'],
    'best_full_brier': best['full_brier'],
    'baseline_full_brier': baseline_brier,
    'improvement': round(baseline_brier - best['full_brier'], 6),
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nSaved to {OUTPUT_PATH}")